## 환경변수 로드

In [ ]:
%pip install -q pymupdf rapidocr_onnxruntime

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## 문서 로드

- 글로벌 EV 시장 동향 및 전망(IEA) 에너지경제연구원 2024.07.15

- 출처: https://eiec.kdi.re.kr/policy/domesticView.do?ac=0000186196&pg=&pp=&search_txt=&issus=&type=&depth1=

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

pdf_filepath = "./data/[24-13-포커스-1] 글로벌 EV 시장 동향 및 전망(IEA).pdf"
loader = PyMuPDFLoader(pdf_filepath, extract_images=True)
docs = loader.load()
print(type(docs),len(docs))

In [ ]:
docs[0].page_content

In [ ]:
docs[0].metadata

## Embedding 성능 비교

In [ ]:
from langchain_ollama import OllamaEmbeddings

nomic_embeddings = OllamaEmbeddings(model="nomic-embed-text")
bge_embeddings = OllamaEmbeddings(model="bge-m3")

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


sentences = [
    "안녕하세요, 오늘은 날씨가 좋습니다.",
    "날씨가 맑아서 기분이 좋아요.",
    "오늘 점심으로 김치찌개를 먹었습니다.",
    "한국의 전통 음식 중 하나는 김치입니다."
]

# 각 모델로 임베딩 생성
nomic_embeds = [nomic_embeddings.embed_query(sent) for sent in sentences]
bge_embeds = [bge_embeddings.embed_query(sent) for sent in sentences]

# 코사인 유사도 계산 함수
def calculate_similarities(embeds):
    return cosine_similarity(embeds)

# 각 모델의 유사도 행렬 계산
nomic_similarities = calculate_similarities(nomic_embeds)
bge_similarities = calculate_similarities(bge_embeds)

# 유사도 비교
for i in range(len(sentences)):
    for j in range(i+1, len(sentences)):
        print(f"\n문장 {i+1}과 문장 {j+1}의 유사도 비교:")
        print(f"Nomic-embed-text: {nomic_similarities[i][j]:.4f}")
        print(f"BGE-M3: {bge_similarities[i][j]:.4f}")

## Chunking

`(1) 재귀적 분할`

In [ ]:
# 문서를 문장 단위로 분리
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n"],
    chunk_size=500,
    chunk_overlap=100,
)

text_chunks = text_splitter.split_documents(docs)
len(text_chunks)

In [ ]:
# chunk 0
print(text_chunks[0].page_content)

In [ ]:
# chunk 1
print(text_chunks[1].page_content)

`(2) 의미적 분할`

In [ ]:
# 문서 분할 
from langchain_experimental.text_splitter import SemanticChunker

semantic_splitter = SemanticChunker(embeddings=bge_embeddings)
semantic_chunks = semantic_splitter.split_documents(text_chunks)

print(f"생성된 청크 수: {len(semantic_chunks)}")

In [ ]:
# chunk 0
print(semantic_chunks[0].page_content)

In [ ]:
# chunk 1
print(semantic_chunks[1].page_content)

In [ ]:
# chunk 2
print(semantic_chunks[2].page_content)

## Indexing

In [ ]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(semantic_chunks, bge_embeddings)

In [ ]:
# 구체적인 사실 관계에 대한 질문 
query = "중국 전기차 시장에서 중소형 모델의 판매 비중은 얼마인가요?"

# 가장 유사도가 높은 문장을 하나만 추출
retriever = vectorstore.as_retriever(search_kwargs={'k': 2})

results = retriever.invoke(query)
print(len(results))
print()

for doc in results:
    print(doc.page_content)
    print("-"*100)

In [ ]:
# 추론이 필요한 질문
query = "중소형 전기차 모델이 가장 많은 판매량을 차지하는 지역은 어디인가요?"

# 가장 유사도가 높은 문장을 하나만 추출
retriever = vectorstore.as_retriever(search_kwargs={'k': 2})

results = retriever.invoke(query)
print(len(results))
print()

for doc in results:
    print(doc.page_content)
    print("-"*100)

## RAG Chain

`(1) Gemma2 활용`

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import ChatOllama


llm = ChatOllama(
    model = "gemma2",
    temperature = 0.2,
    num_predict = 250,
)


# Prompt
template = '''Answer the question based only on the following context.

[Context]
{context}

[Question]
{question}

[Answer (in Korean)]
'''

prompt = ChatPromptTemplate.from_template(template)


def format_docs(docs):
    return '\n\n'.join([d.page_content for d in docs])

# RAG Chain 연결
rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Chain 실행
query = "중국 전기차 시장에서 중소형 모델의 판매 비중은 얼마인가요?"
rag_chain.invoke(query)

In [ ]:
query = "중소형 전기차 모델이 가장 많은 판매량을 차지하는 지역은 어디인가요?"
rag_chain.invoke(query)

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="gemma2-9b-it",
    temperature=0.2,
    max_retries=2,
)

# RAG Chain 연결
rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Chain 실행
query = "글로벌 전기차 산업이 어떻게 재편되고 있나요?"
rag_chain.invoke(query)

`(2) Qwen 2.5 활용`

In [ ]:
llm = ChatOllama(
    model = "qwen2.5",
    temperature = 0.2,
    num_predict = 250,
)

In [ ]:
# RAG Chain 연결
rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Chain 실행
query = "중국 전기차 시장에서 중소형 모델의 판매 비중은 얼마인가요?"
rag_chain.invoke(query)

In [ ]:
query = "중소형 전기차 모델이 가장 많은 판매량을 차지하는 지역은 어디인가요?"
rag_chain.invoke(query)

In [ ]:
query = "글로벌 전기차 산업이 어떻게 재편되고 있나요?"
rag_chain.invoke(query)